# THS 2018 — Weighted OD Matrices (household expansion weights)

Recreates `matrix_10` and `matrix_20` from `THS_2018_MTX.ipynb`, replacing raw trip counts with **household expansion weights** (`wf_new` from `Input/Matrices/households_with_weights.csv`, joined on `HHID`). Each trip contributes its household's `wf_new` weight to the origin–destination cell instead of contributing 1.

Trip-extraction logic is identical to the original notebook:
- slice by `ACT_DAY` = 10 / 20
- order by `INDIVID`, `tourID`, `ACT_ID`
- origin = `taz` of the previous activity, destination = `taz` of the current activity
- leaving time = `EndTime` if `mainActivity == 'Home'`, otherwise `StartTime`
- keep departures between 6:00 and 9:00

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('Input/Matrices/ACTIVITIES_DEC18_corrected.csv')
weights = pd.read_csv('Input/Matrices/households_with_weights.csv')

print(f"activities: {df.shape[0]:,} rows, {df['HHID'].nunique():,} households")
print(f"weights:    {weights.shape[0]:,} households")
weights.head()

activities: 172,529 rows, 5,108 households
weights:    5,108 households


,HHID,TAZ,SuperZone,wf,wf_new
0,2250000572,1803,18,96.875000,17.863711
1,2250000584,1803,18,96.875000,79.546307
2,2250001018,1804,18,70.000000,12.907972
3,2250001030,1804,18,70.000000,10.860661
4,2250001187,1807,18,583.333333,156.928386


In [3]:
# Join the household weight onto every activity record (every HHID has exactly one weight)
df = df.merge(weights[['HHID', 'wf_new']], on='HHID', how='left', validate='many_to_one')
assert df['wf_new'].notna().all(), "some households are missing a weight"
print(f"join OK — all {df['HHID'].nunique():,} households matched")

join OK — all 5,108 households matched


In [4]:
def process_day(df_day, weighted=True):
    # Sort and group
    df_sorted = df_day.sort_values(by=['INDIVID', 'tourID', 'ACT_ID'])

    # Get origin (taz of previous ACT_ID)
    df_sorted['origin'] = df_sorted.groupby(['INDIVID', 'tourID'])['taz'].shift(1)
    df_sorted['destination'] = df_sorted['taz']

    # Determine leaving time with explicit dayfirst=True
    df_sorted['StartTime'] = pd.to_datetime(df_sorted['StartTime'], dayfirst=True)
    df_sorted['EndTime'] = pd.to_datetime(df_sorted['EndTime'], dayfirst=True)

    df_sorted['leaving_time'] = pd.to_datetime(np.where(
        df_sorted['mainActivity'] == 'Home',
        df_sorted['EndTime'],
        df_sorted['StartTime']
    ))

    # Filter for leaving time between 6:00 and 9:00
    mask = (df_sorted['leaving_time'].dt.hour >= 6) & (df_sorted['leaving_time'].dt.hour < 9)
    df_filtered = df_sorted[mask].dropna(subset=['origin', 'destination'])
    # model-area trips only: both ends must have a real TAZ; Default arrival = no reported travel
    df_filtered = df_filtered[(df_filtered['origin'] != 0) & (df_filtered['destination'] != 0)
                              & (df_filtered['MODE_NAME'] != 'Default')]

    if weighted:
        # Weighted matrix: sum of household weights per OD pair
        matrix = pd.crosstab(df_filtered['origin'], df_filtered['destination'],
                             values=df_filtered['wf_new'], aggfunc='sum').fillna(0)
    else:
        # Original behaviour: raw trip counts
        matrix = pd.crosstab(df_filtered['origin'], df_filtered['destination'])
    return matrix

# Slice by ACT_DAY = 10 and 20
df_10 = df[df['ACT_DAY'] == 10].copy()
df_20 = df[df['ACT_DAY'] == 20].copy()

matrix_10_weighted = process_day(df_10, weighted=True)
matrix_20_weighted = process_day(df_20, weighted=True)

print("Weighted matrix for Day 10 shape:", matrix_10_weighted.shape)
print("Weighted matrix for Day 20 shape:", matrix_20_weighted.shape)
matrix_10_weighted.head()

Weighted matrix for Day 10 shape: (656, 706)
Weighted matrix for Day 20 shape: (650, 695)


destination,101,102,103,104,105,106,107,108,109,110,...,4013,4014,4015,4016,4017,4018,4019,4020,4021,4022
origin,,,,,,,,,,,,,,,,,,,,,
101.0,1584.052745,318.680033,0.000000,0.000000,0.000000,0.000000,306.53562,0.0,267.294238,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
102.0,0.000000,329.967623,0.000000,0.000000,0.000000,0.000000,0.00000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
103.0,0.000000,0.000000,60.336448,124.157502,0.000000,0.000000,0.00000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
104.0,0.000000,219.511352,0.000000,740.078027,0.000000,124.157502,0.00000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
105.0,0.000000,0.000000,329.332930,439.022704,3592.984939,305.352854,0.00000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Comparison: raw counts vs weighted totals

The unweighted matrices (recomputed here with the same logic) show sampled trips; the weighted matrices expand them to population-level trips via `wf_new`.

In [5]:
matrix_10_raw = process_day(df_10, weighted=False)
matrix_20_raw = process_day(df_20, weighted=False)

summary = pd.DataFrame({
    'Day 10': [matrix_10_raw.sum().sum(), matrix_10_weighted.sum().sum()],
    'Day 20': [matrix_20_raw.sum().sum(), matrix_20_weighted.sum().sum()],
}, index=['sampled trips (raw count)', 'expanded trips (sum of wf_new)'])
summary.round(1)

,Day 10,Day 20
sampled trips (raw count),14420.0,14197.0
expanded trips (sum of wf_new),2193422.5,2163319.7


In [6]:
# Implied average expansion factor per trip
avg_exp = summary.loc['expanded trips (sum of wf_new)'] / summary.loc['sampled trips (raw count)']
avg_exp.round(2).to_frame('avg expansion factor per trip')

,avg expansion factor per trip
Day 10,152.11
Day 20,152.38


### Row-normalized probability matrices (weighted)

Same normalization as the original notebook, but on the weighted matrices.

In [7]:
prob_matrix_10_weighted = matrix_10_weighted.div(matrix_10_weighted.sum(axis=1), axis=0).fillna(0)
prob_matrix_20_weighted = matrix_20_weighted.div(matrix_20_weighted.sum(axis=1), axis=0).fillna(0)

print("Probability Matrix 10 (weighted, first 5 rows/cols):")
prob_matrix_10_weighted.iloc[:5, :5]

Probability Matrix 10 (weighted, first 5 rows/cols):


destination,101,102,103,104,105
origin,,,,,
101.0,0.35984,0.072393,0.000000,0.000000,0.000000
102.0,0.00000,0.540679,0.000000,0.000000,0.000000
103.0,0.00000,0.000000,0.327038,0.672962,0.000000
104.0,0.00000,0.059796,0.000000,0.201601,0.000000
105.0,0.00000,0.000000,0.037576,0.050092,0.409952


In [8]:
import os
os.makedirs('Output', exist_ok=True)

matrix_10_weighted.to_csv('Output/matrix_10_weighted.csv')
matrix_20_weighted.to_csv('Output/matrix_20_weighted.csv')
prob_matrix_10_weighted.to_csv('Output/prob_matrix_10_weighted.csv')
prob_matrix_20_weighted.to_csv('Output/prob_matrix_20_weighted.csv')

print("Saved:")
for f in ['matrix_10_weighted', 'matrix_20_weighted', 'prob_matrix_10_weighted', 'prob_matrix_20_weighted']:
    print(f" - Output/{f}.csv")

Saved:
 - Output/matrix_10_weighted.csv
 - Output/matrix_20_weighted.csv
 - Output/prob_matrix_10_weighted.csv
 - Output/prob_matrix_20_weighted.csv


### Note on the input data version

The `ACTIVITIES_DEC18_corrected.csv` currently in the repository contains more records than the file used in the original `THS_2018_MTX.ipynb` Colab run: under the original (pre-model-area-filter) trip definition the same logic yields 14,912 unweighted Day-10 trips here vs 11,303 recorded in that notebook's outputs (Day 20: 14,705 vs ~11k). Spot-checked OD cells that existed in both runs match exactly (e.g. 101→101 = 6, 107→107 = 13), confirming the processing is identical and the difference comes from additional survey records in the current file (mostly zones 0, 103–106).